In [ ]:
import tensorflow as tf
import numpy as np
import time
from datasets import load_dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders


vocab_size      = 2700
dim             = 576
n_layers        = 9
n_q_heads       = 9              
n_kv_heads      = 3              
head_dim        = dim // n_q_heads
ff_hidden       = 1728           
seq             = 192
batch           = 48
al_top          = 8
lr              = 2e-4
steps_per_epoch = 150
epochs          = 40
patience        = 10
meta_lr         = 1e-3
inner_lr        = 0.01
maml_task_bs    = 8
maml_n_tasks    = 2

USE_MAML = True
USE_AL   = True


print("loading TinyStories...")
raw_train = load_dataset("roneneldan/TinyStories", split="train")
raw_val   = load_dataset("roneneldan/TinyStories", split="validation")


print("training BPE tokenizer (~2,700 vocab) on TinyStories...")
tok = Tokenizer(models.BPE(unk_token="<unk>"))
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok.decoder       = decoders.ByteLevel()
trainer = trainers.BpeTrainer(
    vocab_size=vocab_size,
    special_tokens=["<pad>", "<unk>", "<bos>", "<eos>"]
)

def tok_iter(n=40000):
    for i, row in enumerate(raw_train):
        if i >= n:
            break
        yield row["text"]

tok.train_from_iterator(tok_iter(), trainer=trainer)
vocab_size = tok.get_vocab_size()
pad_id = tok.token_to_id("<pad>")
bos_id = tok.token_to_id("<bos>")
eos_id = tok.token_to_id("<eos>")
print(f"actual vocab size: {vocab_size}")

def encode_stream(dataset, n_docs):
    ids = []
    for i, row in enumerate(dataset):
        if i >= n_docs:
            break
        e = tok.encode(row["text"]).ids
        ids.extend(e + [eos_id])
    return np.array(ids, dtype=np.int32)

print("encoding train pool...")
train_ids = encode_stream(raw_train, 30000)
print(f"train pool: {len(train_ids):,} tokens")

print("encoding held-out eval pool (validation split — never trained on)...")
eval_ids = encode_stream(raw_val, 4000)
print(f"eval pool: {len(eval_ids):,} tokens\n")

def make_batch(data, n):
    if data is None or len(data) < seq + 2:
        return None, None
    idx = np.random.randint(0, len(data) - seq - 1, n)
    x   = np.stack([data[i:   i+seq  ] for i in idx])
    y   = np.stack([data[i+1: i+seq+1] for i in idx])
    return tf.cast(x, tf.int32), tf.cast(y, tf.int32)


def build_rope_cache(t, dim, base=10000.0):
    inv_freq = 1.0 / (base ** (np.arange(0, dim, 2).astype(np.float32) / dim))
    positions = np.arange(t).astype(np.float32)
    freqs = np.outer(positions, inv_freq)
    emb = np.concatenate([freqs, freqs], axis=-1)
    return tf.constant(np.cos(emb), tf.float32), tf.constant(np.sin(emb), tf.float32)

def rotate_half(x):
    d = x.shape[-1]
    x1, x2 = x[..., :d//2], x[..., d//2:]
    return tf.concat([-x2, x1], axis=-1)

def apply_rope(x, cos, sin):
    cos = cos[tf.newaxis, tf.newaxis, :, :]
    sin = sin[tf.newaxis, tf.newaxis, :, :]
    return x * cos + rotate_half(x) * sin


class RMSNorm(tf.keras.layers.Layer):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.scale = self.add_weight(shape=(dim,), initializer="ones", trainable=True)

    def call(self, x):
        var = tf.reduce_mean(tf.square(x), axis=-1, keepdims=True)
        return x * tf.math.rsqrt(var + self.eps) * self.scale


tok_emb = tf.keras.layers.Embedding(vocab_size, dim)

class DecoderLayer(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.norm2 = RMSNorm(dim)
        self.wq = tf.keras.layers.Dense(n_q_heads * head_dim, use_bias=False)
        self.wk = tf.keras.layers.Dense(n_kv_heads * head_dim, use_bias=False)
        self.wv = tf.keras.layers.Dense(n_kv_heads * head_dim, use_bias=False)
        self.wo = tf.keras.layers.Dense(dim, use_bias=False)
        self.ff1 = tf.keras.layers.Dense(ff_hidden, activation="gelu")
        self.ff2 = tf.keras.layers.Dense(dim)

    def call(self, h, cos, sin, mask):
        b = tf.shape(h)[0]
        t = tf.shape(h)[1]

        hn = self.norm1(h)
        q = tf.transpose(tf.reshape(self.wq(hn), [b, t, n_q_heads, head_dim]), [0, 2, 1, 3])
        k = tf.transpose(tf.reshape(self.wk(hn), [b, t, n_kv_heads, head_dim]), [0, 2, 1, 3])
        v = tf.transpose(tf.reshape(self.wv(hn), [b, t, n_kv_heads, head_dim]), [0, 2, 1, 3])

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        rep = n_q_heads // n_kv_heads
        k = tf.repeat(k, rep, axis=1)
        v = tf.repeat(v, rep, axis=1)

        scores = tf.matmul(q, k, transpose_b=True) / tf.sqrt(tf.cast(head_dim, tf.float32))
        scores += mask
        weights = tf.nn.softmax(scores, axis=-1)
        attn_out = tf.matmul(weights, v)
        attn_out = tf.reshape(tf.transpose(attn_out, [0, 2, 1, 3]), [b, t, n_q_heads * head_dim])
        attn_out = self.wo(attn_out)

        h = h + attn_out
        hn2 = self.norm2(h)
        ff_out = self.ff2(self.ff1(hn2))
        h = h + ff_out
        return h

decoder_layers = [DecoderLayer() for _ in range(n_layers)]
final_norm = RMSNorm(dim)

def causal_mask(t):
    m = 1 - tf.linalg.band_part(tf.ones((t, t)), -1, 0)
    return m * -1e9

_cos_cache, _sin_cache = build_rope_cache(seq, head_dim)

def forward(x, training=False):
    t = tf.shape(x)[1]
    h = tok_emb(x)
    mask = causal_mask(t)
    cos = _cos_cache[:t]
    sin = _sin_cache[:t]
    for layer in decoder_layers:
        h = layer(h, cos, sin, mask)
    h = final_norm(h)
    logits = tf.matmul(h, tok_emb.embeddings, transpose_b=True) 
    return logits

def all_vars():
    v = list(tok_emb.trainable_variables) + list(final_norm.trainable_variables)
    for layer in decoder_layers:
        v += layer.trainable_variables
    return v

def loss_fn(x, y, training=False):
    logit = forward(x, training=training)
    return tf.reduce_mean(
        tf.keras.losses.sparse_categorical_crossentropy(y, logit, from_logits=True))

def eval_loss_full(data, bs=32, max_batches=25):
    losses = []
    n = len(data) - seq - 1
    starts = np.random.choice(n, size=min(n, bs * max_batches), replace=False)
    for i in range(0, len(starts) - bs, bs):
        idx = starts[i:i+bs]
        x = tf.cast(np.stack([data[j:j+seq]     for j in idx]), tf.int32)
        y = tf.cast(np.stack([data[j+1:j+seq+1] for j in idx]), tf.int32)
        losses.append(loss_fn(x, y, training=False).numpy())
    return float(np.mean(losses))

def active_query(pool_x, pool_y, n, mini=16):
    all_ent = []
    for i in range(0, len(pool_x), mini):
        mb = pool_x[i:i+mini]
        prob = tf.nn.softmax(forward(mb, training=False), axis=-1)
        ent = tf.reduce_mean(-tf.reduce_sum(prob * tf.math.log(prob + 1e-9), axis=-1), axis=-1)
        all_ent.append(ent)
    all_ent = tf.concat(all_ent, axis=0)
    idx = tf.argsort(all_ent, direction="DESCENDING")[:n]
    return tf.gather(pool_x, idx), tf.gather(pool_y, idx)

def maml_step(tasks):
    vars = all_vars()
    meta_grads = [tf.zeros_like(v) for v in vars]

    for sx, sy, qx, qy in tasks:
        with tf.GradientTape() as t1:
            s_loss = loss_fn(sx, sy, training=True)
        ig = t1.gradient(s_loss, vars)
        orig = [v.numpy() for v in vars]

        for v, g in zip(vars, ig):
            if g is not None:
                v.assign_sub(inner_lr * tf.convert_to_tensor(g))

        with tf.GradientTape() as t2:
            q_loss = loss_fn(qx, qy, training=True)
        og = t2.gradient(q_loss, vars)

        for v, o in zip(vars, orig):
            v.assign(o)

        for j, g in enumerate(og):
            if g is not None:
                meta_grads[j] += tf.convert_to_tensor(g)

    return [g / len(tasks) for g in meta_grads]

def generate(seed, steps=150, temp=0.8, top_k=40, rep_penalty=1.3, rep_window=20):
    ids = tok.encode(seed).ids
    ctx = tf.constant([ids[-seq:]], dtype=tf.int32)
    out_ids = list(ids)
    recent = []
    for _ in range(steps):
        logit = forward(ctx, training=False)[0, -1].numpy().astype(np.float64) / temp
        for tid in set(recent[-rep_window:]):
            logit[tid] /= rep_penalty
        k = min(top_k, vocab_size)
        top_idx = np.argpartition(logit, -k)[-k:]
        top_vals = logit[top_idx]
        probs = np.exp(top_vals - top_vals.max())
        probs /= probs.sum()
        next_id = int(np.random.choice(top_idx, p=probs))
        if next_id == eos_id:
            break
        recent.append(next_id)
        out_ids.append(next_id)
        ctx = tf.concat([ctx[:, 1:], [[next_id]]], axis=1) if ctx.shape[1] >= seq else \
              tf.concat([ctx, [[next_id]]], axis=1)
    return tok.decode(out_ids)


_ = forward(tf.zeros([1, seq], dtype=tf.int32))
opt = tf.keras.optimizers.Adam(lr)
best_eval, no_improve = float('inf'), 0

n_params = sum(int(np.prod(v.shape)) for v in all_vars())
print(f"model params: {n_params:,}")
print(f"MAML={USE_MAML}  ActiveLearning={USE_AL}")
print("Training: TinyStories (~30M params, single GPU) + honest held-out eval\n")

for ep in range(epochs):
    ep_start = time.time()
    step_losses = []

    for step in range(steps_per_epoch):
        rx, ry = make_batch(train_ids, batch)

        if USE_AL:
            pool_x, pool_y = make_batch(train_ids, 64)
            al_x, al_y = active_query(pool_x, pool_y, al_top, mini=16)
            tx = tf.concat([rx, al_x], axis=0)
            ty = tf.concat([ry, al_y], axis=0)
        else:
            tx, ty = rx, ry

        with tf.GradientTape() as tape:
            tr_loss = loss_fn(tx, ty, training=True)
        grads = tape.gradient(tr_loss, all_vars())
        opt.apply_gradients(zip(grads, all_vars()))
        step_losses.append(tr_loss.numpy())

        if step == 0 and (np.isnan(tr_loss.numpy()) or tr_loss.numpy() == 0.0):
            print("  ⚠ suspicious loss on first step (nan or exactly 0)")
        if step % 25 == 0:
            print(f"  step {step}/{steps_per_epoch}  loss={tr_loss.numpy():.4f}")

    avg_train = np.mean(step_losses)

    if USE_MAML:
        tasks = [(*make_batch(train_ids, maml_task_bs),
                  *make_batch(train_ids, maml_task_bs)) for _ in range(maml_n_tasks)]
        mg = maml_step(tasks)
        for v, g in zip(all_vars(), mg):
            v.assign_sub(meta_lr * g)

    ev_loss = eval_loss_full(eval_ids)
    ppl = float(np.exp(ev_loss))
    ep_time = time.time() - ep_start
    print(f"epoch {ep+1}/{epochs}  train={avg_train:.4f}  eval={ev_loss:.4f}  perplexity={ppl:.1f}  ({ep_time:.0f}s)")

    if ev_loss < best_eval:
        best_eval, no_improve = ev_loss, 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"\nearly stop at epoch {ep+1}  best_eval={best_eval:.4f}")
            break


print("\n── generated text ──")
print(generate("Once upon a time"))
print()
print(generate("The little dog"))
print()
print(generate("One day, a girl"))

print("\n── simple understanding probes ──")

probes = [
    "Tom was very hungry, so he",
    "It started raining, so Mia decided to",
    "The ball was red. Sam picked up the",
    "Lily was sad because she lost her toy. Then her mom",
]
for p in probes:
    print(f"\nPrompt: {p}")
    print(generate(p, steps=60, temp=0.7, top_k=20))

loading TinyStories...


training BPE tokenizer (~2,700 vocab) on TinyStories...
actual vocab size: 2700
encoding train pool...
train pool: 7,075,135 tokens
encoding held-out eval pool (validation split — never trained on)...
eval pool: 869,770 tokens

model params: 27,465,408
MAML=True  ActiveLearning=True
Training: TinyStories (~30M params, single GPU) + honest held-out eval

  step 0/150  loss=8.1121
  step 25/150  loss=5.9613
  step 50/150  loss=5.3298
  step 75/150  loss=4.8173
  step 100/150  loss=4.5486
  step 125/150  loss=4.2418
epoch 1/40  train=5.0566  eval=4.0338  perplexity=56.5  (356s)
  step 0/150  loss=4.0504
  step 25/150  loss=4.0130
  step 50/150  loss=3.7235
  step 75/150  loss=3.7734
  step 100/150  loss=3.5925
  step 125/150  loss=3.4666
epoch 2/40  train=3.6988  eval=3.4730  perplexity=32.2  (358s)
  step 0/150  loss=3.5447
  step 25/150  loss=3.4011
  step 50/150  loss=3.3318
  step 75/150  loss=3.3092
  step 100/150  loss=3.3191
  step 125/150  loss=3.1312
epoch 3/40  train=3.2838  eva

In [ ]:
import numpy as np
import os

os.makedirs("tiny_storyteller_30m", exist_ok=True)


tok.save("tiny_storyteller_30m/tokenizer.json")


weights = {}
for i, v in enumerate(all_vars()):
    weights[f"var_{i}__{v.name}"] = v.numpy()
np.savez("tiny_storyteller_30m/weights.npz", **weights)


import json
config = dict(
    vocab_size=vocab_size, dim=dim, n_layers=n_layers,
    n_q_heads=n_q_heads, n_kv_heads=n_kv_heads, head_dim=head_dim,
    ff_hidden=ff_hidden, seq=seq
)
with open("tiny_storyteller_30m/config.json", "w") as f:
    json.dump(config, f, indent=2)

print("saved to tiny_storyteller_30m/")
print(f"weights.npz size: {os.path.getsize('tiny_storyteller_30m/weights.npz') / 1e6:.1f} MB")


import shutil
shutil.make_archive("tiny_storyteller_30m", "zip", "tiny_storyteller_30m")
print("zipped: tiny_storyteller_30m.zip")


try:
    from google.colab import files
    files.download("tiny_storyteller_30m.zip")
except ImportError:

    print("find tiny_storyteller_30m.zip in the Output tab of this notebook (right sidebar) and click Download.")

saved to tiny_storyteller_30m/
weights.npz size: 109.9 MB
zipped: tiny_storyteller_30m.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>